# Quick Training Walkthrough

Interactive demonstration of the hybrid DDM-FPM model.
Uses synthetic data with reduced epochs for a quick test.

In [ ]:
import os
os.environ['JAX_ENABLE_X64'] = '1'

# Run from repo root
os.chdir(os.path.join(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

## 1. Generate Synthetic Data

In [ ]:
from scripts.make_synthetic_data import make_dataset

df = make_dataset()
df.head()

## 2. Load and Inspect Data

In [ ]:
import numpy as np
import jax.numpy as jnp
from src.data import load_training_data, build_features, prestack_data
from src.train import DEFAULT_CONFIG

cfg = {**DEFAULT_CONFIG, 'max_epochs': 50}

normal_inputs = load_training_data()
print(f"\nFeed nC4 range: {min(inp['z_F'][3] for inp in normal_inputs)*100:.1f} - "
      f"{max(inp['z_F'][3] for inp in normal_inputs)*100:.1f} mol%")
print(f"LPG nC4 range: {min(inp['xD_actual'][3] for inp in normal_inputs)*100:.1f} - "
      f"{max(inp['xD_actual'][3] for inp in normal_inputs)*100:.1f} mol%")

## 3. Prestack & Normalize

In [ ]:
features_np = build_features(normal_inputs)
feat_mean = features_np.mean(axis=0)
feat_std = features_np.std(axis=0) + 1e-8
features_normed = (features_np - feat_mean) / feat_std

data = prestack_data(normal_inputs, cfg['n_stages'], cfg['delta_p'])
data['features_normed'] = jnp.array(features_normed)
data['feat_mean'] = feat_mean
data['feat_std'] = feat_std

print(f"Features shape: {features_np.shape}")
print(f"T_init shape: {data['T_init'].shape}")

## 4. Train (50 epochs, demo)

JIT compilation takes ~60-120s on first call. Subsequent epochs are fast.

In [ ]:
from src.train import train

best_params, loss_history = train(
    data, cfg=cfg, seed=42, save_dir='results/notebook_demo'
)

## 5. Plot Loss Curve

In [ ]:
import matplotlib.pyplot as plt

epochs = range(len(loss_history))
total = [h[0] for h in loss_history]
comp = [h[1] for h in loss_history]
tvp = [h[2] for h in loss_history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, total, label='Total', linewidth=2)
ax.plot(epochs, comp, label='Composition', linewidth=1, alpha=0.7)
ax.plot(epochs, tvp, label='TVP', linewidth=1, alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MAPE)')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Inspect Learned Corrections

In [ ]:
from src.hybrid import ddm_forward, PARAM_LABELS

print(f"{'Day':<12s} {'q':>6s} {'dTVP':>8s}  delta_kij")
print("-" * 70)
for i, inp in enumerate(normal_inputs[:5]):
    feat_n = jnp.array(features_normed[i])
    dkij, q, dtvp = ddm_forward(best_params, feat_n)
    dkij_str = ', '.join(f'{float(dkij[k]):+.4f}' for k in range(6))
    print(f"{inp['date_str']:<12s} {float(q):6.3f} {float(dtvp):+8.4f}  [{dkij_str}]")
print(f"... ({len(normal_inputs)} days total)")